# 🎨 Image Restoration Dual-Model Training - Colab

This notebook supports **Deterministic (SwinIR)** and **Generative (SD-LoRA)** restoration with automatic **Benchmarking**.

### ✅ Requirements:
Ensure you have uploaded **ALL THREE** `restoration_code.zip`, `processed_data.zip`, and `processed_sd.zip` to the root of your Google Drive.

### 1. Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

# Create directories for weights and validation logs
!mkdir -p /content/drive/MyDrive/weights/swinir_checkpoints
!mkdir -p /content/drive/MyDrive/weights/sd_lora_val_images

### 2. Smart Extract: Code & Data

In [ ]:
# 1. Extract Source Code (Fastest)
print("📁 Extracting Code...")
!unzip -q /content/drive/MyDrive/restoration_code.zip -d /content/


In [ ]:
# 2. Extract Large Datasets
if os.path.exists('/content/drive/MyDrive/processed_data.zip'):
    print("📊 Extracting Standard Processed Dataset...")
    !unzip -q /content/drive/MyDrive/processed_data.zip -d /content/



In [ ]:
if os.path.exists('/content/drive/MyDrive/processed_sd.zip'):
    print("🎨 Extracting SD-Optimized Dataset...")
    !unzip -q /content/drive/MyDrive/processed_sd.zip -d /content/
else:
    print("⚠️ 'processed_sd.zip' not found. Run prepare_lite_sd.sh if needed.")


In [ ]:
%cd /content/training

# Install dependencies (fast cloud install)
!pip install -q -r requirements.txt
!pip install -q xformers peft accelerate diffusers transformers controlnet_aux controlnet_aux controlnet_aux

In [ ]:
!pip install mediapipe==0.10.14

### 3. Data Preparation (ONLY if processed_data.zip was missing)
This generates clean/damaged pairs and separates them into `train` and `test` sets automatically.

In [ ]:
# !python scripts/prepare_data.py --input ./datasets/raw_lite \
#                               --output ./datasets/processed \
#                               --seed 42 \
#                               --split_ratio 0.9 \
#                               --multiplier 1

### 4. Track 1: SwinIR-Light Training (Sanity Test)
Run a quick 1-epoch test to ensure the dataset is loaded correctly and CUDA is working before the full run.

In [ ]:
!python scripts/train_swinir.py --clean_dir ./datasets/processed/train/clean \
                                --damaged_dir ./datasets/processed/train/damaged \
                                --epochs 1 \
                                --batch_size 2 \
                                --weight_path /content/drive/MyDrive/weights/swinir_restoration_test.pth

### 4.5 Track 1: SwinIR-Light Training (Full Run)
The model will periodically evaluate on the test set and save the best PSNR weights.

In [ ]:
!python scripts/train_swinir.py --clean_dir ./datasets/processed/train/clean \
                                --damaged_dir ./datasets/processed/train/damaged \
                                --val_clean_dir ./datasets/processed/test/clean \
                                --val_damaged_dir ./datasets/processed/test/damaged \
                                --epochs 100 \
                                --batch_size 16 \
                                --patch_size 128 \
                                --weight_path /content/drive/MyDrive/weights/swinir_restoration_colab.pth \
                                --resume

### 5. Track 2: SD-LoRA Transfer Training (Sanity Test)
Run a quick 1-epoch test with a tiny SD model to ensure the pipeline works.

In [ ]:
!python scripts/train_sd_lora.py --image_dir ./datasets/processed/train/clean \
                                 --output_dir /content/drive/MyDrive/weights/sd_lora_antique_test \
                                 --epochs 1 \
                                 --batch_size 1 \
                                 --model_id hf-internal-testing/tiny-stable-diffusion-torch

### 5.5 Track 2: SD-LoRA Transfer Training (Full Run)
Check `/content/drive/MyDrive/weights/sd_lora_val_images` to see restoration progress.

In [ ]:
!python scripts/train_sd_lora.py --image_dir ./datasets/processed/train/clean \
                                 --output_dir /content/drive/MyDrive/weights/sd_lora_antique_colab \
                                 --val_dir /content/drive/MyDrive/weights/sd_lora_val_images \
                                 --epochs 20 \
                                 --batch_size 4 \
                                 --resume

### 6. Track 3: Benchmarking
Evaluate both networks computationally against the ground truth, and generate an image grid comparing the outputs.

In [ ]:
### 6.1 Evaluate Deterministic SwinIR Model
print("📊 Running Quantitative Metrics (PSNR/SSIM/MSE/NRMSE) for SwinIR...")
!python scripts/evaluate.py --clean_dir ./datasets/processed/test/clean \
                            --damaged_dir ./datasets/processed/test/damaged \
                            --weights /content/drive/MyDrive/weights/swinir_restoration_colab.pth


In [ ]:
### 6.2 Evaluate Generative SD-LoRA Model
print("📊 Running Quantitative Metrics for SD-LoRA...")
print("(Note: This generates the full dataset using ControlNet+LoRA so it takes a little longer!)")
!python scripts/evaluate.py --clean_dir ./datasets/processed/test/clean \
                            --damaged_dir ./datasets/processed/test/damaged \
                            --lora_path /content/drive/MyDrive/weights/sd_lora_antique_colab/latest


In [ ]:
### 6.3 Generate Visual Side-by-Side Comparison Grid
print("🖼️ Generating visual comparison grid...")
!python scripts/test_restoration.py --test_clean_dir ./datasets/processed/test/clean \
                                    --test_damaged_dir ./datasets/processed/test/damaged \
                                    --swinir_weights /content/drive/MyDrive/weights/swinir_restoration_colab.pth \
                                    --lora_path /content/drive/MyDrive/weights/sd_lora_antique_colab/latest \
                                    --output_dir ./test_results \
                                    --num_test 10


In [ ]:
# Display results
from IPython.display import Image, display
import glob
grids = glob.glob('./test_results/*.png')
if grids:
    display(Image(filename=grids[0]))

In [ ]:
# Display all generated result grids
from IPython.display import Image, display
import glob
grids = sorted(glob.glob('./test_results/*.png'))
if grids:
    for grid in grids:
        display(Image(filename=grid))
else:
    print("No test results found.")


### 7. Iterative/Recursive Restoration Testing
Pass test images multiple times through the models to see the compound effect (e.g., n=10 passes).

In [ ]:
### 7.1 Run Recursive Test (N=10 passes)
print("🔄 Running recursive inference (this will take a while for SD-LoRA!)...")
!python scripts/test_recursive.py --test_damaged_dir ./datasets/processed/test/damaged \
                                  --swinir_weights /content/drive/MyDrive/weights/swinir_restoration_colab.pth \
                                  --lora_path /content/drive/MyDrive/weights/sd_lora_antique_colab/latest \
                                  --output_dir ./recursive_results \
                                  --num_test 2 \
                                  --passes 10


In [ ]:
# Display the Iterative GIF/Grid Results
from IPython.display import Image, display
import glob
print("SwinIR Effect:")
swinir_gifs = sorted(glob.glob('./recursive_results/*_swinir_effect.gif'))
for gif in swinir_gifs:
    display(Image(filename=gif))

print("SD-LoRA Effect:")
sd_gifs = sorted(glob.glob('./recursive_results/*_sd_effect.gif'))
for gif in sd_gifs:
    display(Image(filename=gif))


In [ ]:
### 7.2 Run Recursive Test SUPER (N=30 passes)
print("🔄 Running recursive inference (this will take a while for SD-LoRA!)...")
!python scripts/test_recursive.py --test_damaged_dir ./datasets/processed/test/damaged \
                                  --swinir_weights /content/drive/MyDrive/weights/swinir_restoration_colab.pth \
                                  --lora_path /content/drive/MyDrive/weights/sd_lora_antique_colab/latest \
                                  --output_dir ./recursive_results_super \
                                  --num_test 2 \
                                  --passes 30

In [ ]:
# Display the Iterative GIF/Grid Results
from IPython.display import Image, display
import glob
print("SwinIR Effect:")
swinir_gifs = sorted(glob.glob('./recursive_results_super/*_swinir_effect.gif'))
for gif in swinir_gifs:
    display(Image(filename=gif))

print("SD-LoRA Effect:")
sd_gifs = sorted(glob.glob('./recursive_results_super/*_sd_effect.gif'))
for gif in sd_gifs:
    display(Image(filename=gif))


### 8. Live Interactive Restoration: Manual Damage & SD-LoRA Recovery
This section allows you to pick an image, apply damage to a **specific position**, and see how SD-LoRA + ControlNet restores it.

In [ ]:
# @title 🛠️ Restoration Sandbox Configuration
damage_x = 100 # @param {type:"slider", min:0, max:512, step:1}
damage_y = 100 # @param {type:"slider", min:0, max:512, step:1}
damage_width = 300 # @param {type:"slider", min:10, max:512, step:1}
damage_height = 300 # @param {type:"slider", min:10, max:512, step:1}
damage_type = "mold" # @param ["scratch", "mold", "yellowed", "fold", "solid", "average", "heavy_blur", "blackout"]
solid_color_hex = "#ffffff" # @param {type:"color"}
prompt_text = "a perfectly restored antique photo, high detail, sharp, cinematic lighting" # @param {type:"string"}

import cv2
import numpy as np
import torch
import random
from PIL import Image
import matplotlib.pyplot as plt
from diffusers import StableDiffusionControlNetInpaintPipeline, ControlNetModel
from controlnet_aux import CannyDetector
from peft import PeftModel
from glob import glob
import os

def hex_to_rgb(value):
    value = value.lstrip('#')
    return tuple(int(value[i:i+2], 16) for i in (4, 2, 0)) # BGR

def apply_advanced_damage(img, pos=(128, 128, 256, 256), d_type='mold', color=(255, 255, 255)):
    damaged = img.copy().astype(np.float32) / 255.0
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    x, y, w, h = pos
    h_img, w_img, _ = damaged.shape
    x2, y2 = min(x+w, w_img), min(y+h, h_img)
    
    # Sub-patch for manipulation
    patch = damaged[y:y2, x:x2, :].copy()
    ph, pw, _ = patch.shape
    if ph == 0 or pw == 0: return img, mask
    
    if d_type == 'scratch':
        # Draw white thin lines
        for _ in range(random.randint(5, 15)):
            p1 = (random.randint(0, pw), random.randint(0, ph))
            p2 = (random.randint(0, pw), random.randint(0, ph))
            cv2.line(patch, p1, p2, (1, 1, 1), random.randint(1, 2))
            
    elif d_type == 'mold':
        # Organic elliptical stains (rust/mold color)
        for _ in range(random.randint(3, 7)):
            center = (random.randint(0, pw), random.randint(0, ph))
            axes = (random.randint(10, pw//2), random.randint(10, ph//2))
            stain = np.zeros_like(patch)
            cv2.ellipse(stain, center, axes, random.randint(0, 360), 0, 360, (0.2, 0.3, 0.5), -1)
            stain = cv2.GaussianBlur(stain, (21, 21), 0)
            patch = patch * (1 - stain) + stain
            
    elif d_type == 'yellowed':
        # Kill blue, keep red/green for sepia/yellowing
        patch[:, :, 0] *= 0.4 # Blue
        patch[:, :, 1] *= 0.9 # Green
        patch[:, :, 2] *= 1.1 # Red boost
        patch = np.clip(patch * 0.8 + 0.1, 0, 1)
        
    elif d_type == 'fold':
        # A thick "paper fold" artifact
        is_horiz = random.random() > 0.5
        if is_horiz:
            fy = random.randint(0, ph)
            cv2.line(patch, (0, fy), (pw, fy), (0.9, 0.9, 0.9), random.randint(3, 8))
        else:
            fx = random.randint(0, pw)
            cv2.line(patch, (fx, 0), (fx, ph), (0.9, 0.9, 0.9), random.randint(3, 8))
            
    elif d_type == 'solid':
        patch[:, :, :] = np.array(color) / 255.0
    elif d_type == 'average':
        patch[:, :, :] = np.mean(damaged, axis=(0, 1))
    elif d_type == 'blackout':
        patch[:, :, :] = 0.0
    elif d_type == 'heavy_blur':
        patch = cv2.GaussianBlur(patch, (99, 99), 0)
    
    damaged[y:y2, x:x2, :] = patch
    mask[y:y2, x:x2] = 255
    return (damaged * 255.0).astype(np.uint8), mask

def run_live_test(image_path):
    # Load and prep
    clean = cv2.imread(image_path)
    clean = cv2.resize(clean, (512, 512))
    
    # Damage and Get Mask
    bgr_color = hex_to_rgb(solid_color_hex)
    pos_rect = (damage_x, damage_y, damage_width, damage_height)
    damaged, mask = apply_advanced_damage(clean, pos_rect, damage_type, bgr_color)
    
    # Structure Skeleton (Using CLEAN for structure, showing generative fill power)
    canny_raw = Image.fromarray(cv2.cvtColor(clean, cv2.COLOR_BGR2RGB))
    control_image = canny_detector(canny_raw, low_threshold=100, high_threshold=200)
    
    init_image = Image.fromarray(cv2.cvtColor(damaged, cv2.COLOR_BGR2RGB))
    mask_image = Image.fromarray(mask)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    autocast_device = "cuda" if torch.cuda.is_available() else "cpu"
    with torch.autocast(autocast_device):
        restored = pipe(
            prompt=prompt_text, 
            image=init_image, 
            mask_image=mask_image, 
            control_image=control_image,
            strength=0.9, 
            controlnet_conditioning_scale=1.0, 
            num_inference_steps=30
        ).images[0]
    
    restored_np = np.array(restored)
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(cv2.cvtColor(clean, cv2.COLOR_BGR2RGB)); axes[0].set_title("1. Original Clean")
    
    vis_damaged = damaged.copy()
    cv2.rectangle(vis_damaged, (pos_rect[0], pos_rect[1]), 
                  (pos_rect[0]+pos_rect[2], pos_rect[1]+pos_rect[3]), (0, 0, 255), 2)
    axes[1].imshow(cv2.cvtColor(vis_damaged, cv2.COLOR_BGR2RGB)); axes[1].set_title(f"2. Applied: {damage_type}")
    
    axes[2].imshow(restored_np); axes[2].set_title("3. SD-LoRA Restoration Result")
    for ax in axes: ax.axis('off')
    plt.show()

# Get images
test_images = sorted(glob("./datasets/processed/test/clean/*.*"))
if test_images:
    run_live_test(test_images[0])
else:
    print("No test images found!")
